# Embeddings and Vector Store
**Author:** Juan Esteban Agudelo Ortiz  
**Email:** juan.es.agor@gmail.com

---

This notebook builds an embedding and indexing pipeline that converts
text chunks into dense vector representations and stores them in a
vector database for efficient semantic search. The input is a list of
chunks produced by the chunking pipeline notebook.

An embedding model maps each chunk to a vector $\mathbf{e} \in \mathbb{R}^d$
where $d$ is the embedding dimension. Semantic similarity between two
chunks is measured by cosine similarity between their embeddings:

$$\text{sim}(\mathbf{e}_i, \mathbf{e}_j) = \frac{\mathbf{e}_i \cdot \mathbf{e}_j}{\|\mathbf{e}_i\| \|\mathbf{e}_j\|}$$

ChromaDB stores the embeddings and supports approximate nearest neighbor
search, returning the $k$ most similar chunks to a query without
comparing against every stored vector.

### Limitations
1. Embeddings are model-specific; chunks indexed with one model cannot
   be searched with a different model.
2. Cosine similarity measures semantic relatedness, not factual correctness;
   a retrieved chunk may be topically relevant but not answer the query.
3. ChromaDB persistence is local; the index must be rebuilt if the
   chunks or embedding model change.
4. The default embedding model is multilingual (paraphrase-multilingual-MiniLM-L12-v2)
   and supports Spanish and English. Switching to a monolingual model will degrade
   performance on Spanish text.

## 0. Install dependencies

In [1]:
# Run only once
# !pip install llama-index-core llama-index-embeddings-huggingface chromadb llama-index-vector-stores-chroma

## 1. Imports and configuration

In [2]:
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional
from enum import Enum

import chromadb
from llama_index.core import VectorStoreIndex, StorageContext, Document
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.chroma import ChromaVectorStore

# --- Directory setup ---
BASE_DIR    = Path("..")
UPLOADS_DIR = BASE_DIR / "data" / "uploads"
OUT_DIR     = BASE_DIR / "outputs"
INDEX_DIR   = BASE_DIR / "data" / "index"

UPLOADS_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
INDEX_DIR.mkdir(parents=True, exist_ok=True)

# --- Reproduce IngestedDocument from notebook 1 ---
class InputType(Enum):
    PDF           = "pdf"
    HANDWRITTEN   = "handwritten_image"
    REFERENCE_IMG = "reference_image"
    PLAIN_TEXT    = "plain_text"

@dataclass
class IngestedDocument:
    input_type       : InputType
    text             : str
    reference_images : list = field(default_factory=list)
    source_path      : Optional[Path] = None

print(f"Uploads dir : {UPLOADS_DIR.resolve()}")
print(f"Index dir   : {INDEX_DIR.resolve()}")
print("Configuration ready ✓")

Uploads dir : /home/juanessao/Documents/Datos_de_Ciencia/playlists/hugging-face-hackaton/flashcard-generator-slm/data/uploads
Index dir   : /home/juanessao/Documents/Datos_de_Ciencia/playlists/hugging-face-hackaton/flashcard-generator-slm/data/index
Configuration ready ✓


## 2. Embedding model

The embedding model maps each text chunk to a dense vector
$\mathbf{e} \in \mathbb{R}^d$ where $d = 384$ for
`paraphrase-multilingual-MiniLM-L12-v2`.

The model is initialized once and reused across all indexing and search
operations. Using a different model for indexing and searching produces
incoherent results because the vector spaces are incompatible.

In [3]:
# Initialize embedding model once — downloads on first run (~120MB)
EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

embed_model = HuggingFaceEmbedding(
    model_name = EMBEDDING_MODEL_NAME,
    device     = "cpu",
)

# Verify embedding dimension
test_embedding = embed_model.get_text_embedding("test")
print(f"Model         : {EMBEDDING_MODEL_NAME}")
print(f"Embedding dim : {len(test_embedding)}")
print("Embedding model ready ✓")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model         : sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Embedding dim : 384
Embedding model ready ✓


## 3. Vector store with ChromaDB

ChromaDB stores embeddings in a **collection** — a named container
analogous to a table in a relational database. Each entry in the
collection contains:

- The embedding vector $\mathbf{e} \in \mathbb{R}^d$
- The original chunk text
- Metadata: source document, chunk index, chunking strategy

ChromaDB uses **HNSW** (Hierarchical Navigable Small World) as its
default index structure. HNSW organizes vectors in a layered graph
where each node connects to its nearest neighbors. Search traverses
the graph from coarse to fine layers, finding approximate nearest
neighbors in $O(\log n)$ instead of the $O(n)$ required by brute-force
comparison.

In [4]:
def create_vector_store(
    collection_name: str,
    persist_dir: Path,
) -> tuple[chromadb.Collection, ChromaVectorStore]:
    """
    Create or load a ChromaDB collection and its LlamaIndex wrapper.

    Parameters
    ----------
    collection_name : str
        Name of the ChromaDB collection.
    persist_dir : Path
        Directory where ChromaDB persists the index to disk.

    Returns
    -------
    tuple[chromadb.Collection, ChromaVectorStore]
        The ChromaDB collection and its LlamaIndex wrapper.
    """
    chroma_client = chromadb.PersistentClient(path=str(persist_dir))
    collection    = chroma_client.get_or_create_collection(collection_name)
    vector_store  = ChromaVectorStore(chroma_collection=collection)

    print(f"Collection    : {collection_name}")
    print(f"Persist dir   : {persist_dir.resolve()}")
    print(f"Existing docs : {collection.count()}")

    return collection, vector_store


# Quick test
collection, vector_store = create_vector_store(
    collection_name = "test_collection",
    persist_dir     = INDEX_DIR / "test",
)

Collection    : test_collection
Persist dir   : /home/juanessao/Documents/Datos_de_Ciencia/playlists/hugging-face-hackaton/flashcard-generator-slm/data/index/test
Existing docs : 0


## 4. Indexing pipeline

The indexing pipeline converts a list of chunks into a searchable
`VectorStoreIndex`. Each chunk is embedded and stored in ChromaDB
with its metadata.

The pipeline accepts chunks produced by any of the three chunking
strategies from the chunking pipeline notebook. The chunking strategy
is stored as metadata alongside each chunk, enabling traceability of
retrieved results.

In [5]:
def build_index(
    chunks: list[dict],
    vector_store: ChromaVectorStore,
    embed_model: HuggingFaceEmbedding,
) -> VectorStoreIndex:
    """
    Build a VectorStoreIndex from a list of chunks.

    Parameters
    ----------
    chunks : list[dict]
        Chunks produced by any chunking strategy. Each chunk must have
        'text', 'index', and 'strategy' keys.
    vector_store : ChromaVectorStore
        ChromaDB vector store to index into.
    embed_model : HuggingFaceEmbedding
        Embedding model to use for indexing.

    Returns
    -------
    VectorStoreIndex
        Searchable index over the chunks.
    """
    documents = [
        Document(
            text     = chunk["text"],
            metadata = {
                "chunk_index" : chunk["index"],
                "strategy"    : chunk["strategy"],
            },
        )
        for chunk in chunks
    ]

    storage_context = StorageContext.from_defaults(
        vector_store = vector_store,
    )

    index = VectorStoreIndex.from_documents(
        documents,
        storage_context = storage_context,
        embed_model     = embed_model,
        show_progress   = True,
    )

    print(f"Indexed {len(documents)} chunks ✓")
    return index


print("build_index defined ✓")

build_index defined ✓


## 5. Similarity search

Given a query string, the search pipeline:
1. Embeds the query using the same model used for indexing
2. Retrieves the $k$ most similar chunks by cosine similarity
3. Returns the chunks with their metadata and similarity scores

The query embedding $\mathbf{e}_q$ is compared against all stored
embeddings $\{\mathbf{e}_1, \ldots, \mathbf{e}_n\}$ using HNSW.
The top $k$ chunks with highest cosine similarity are returned:

$$\text{top-}k = \underset{i}{\text{arg top-}k}\ \text{sim}(\mathbf{e}_q, \mathbf{e}_i)$$

The choice of $k$ involves a tradeoff: larger $k$ provides more context
to the language model but increases the risk of including irrelevant chunks.
A typical value is $k \in [3, 5]$.

In [6]:
def search(
    index: VectorStoreIndex,
    query: str,
    k: int = 3,
    embed_model: HuggingFaceEmbedding = None,
) -> list[dict]:
    """
    Retrieve the k most similar chunks to a query.

    Parameters
    ----------
    index : VectorStoreIndex
        Searchable index built by build_index.
    query : str
        Query string to search for.
    k : int
        Number of chunks to retrieve. Default 3.
        Keep k * mean_chunk_size well below the model context window.
    embed_model : HuggingFaceEmbedding
        Embedding model. Must be the same model used for indexing.

    Returns
    -------
    list[dict]
        Retrieved chunks with keys:
        - 'text'     : str   — chunk text
        - 'score'    : float — cosine similarity score
        - 'strategy' : str   — chunking strategy
        - 'index'    : int   — chunk position in original document
    """
    retriever = index.as_retriever(
        similarity_top_k = k,
        embed_model      = embed_model,
    )

    nodes = retriever.retrieve(query)

    return [
        {
            "text"     : node.text,
            "score"    : node.score,
            "strategy" : node.metadata.get("strategy", "unknown"),
            "index"    : node.metadata.get("chunk_index", -1),
        }
        for node in nodes
    ]


print("search defined ✓")

search defined ✓


## 6. Persistence to disk

ChromaDB persists the index automatically to `INDEX_DIR` via
`PersistentClient`. However, if the embedding model changes, the
stored vectors are incompatible with the new model and the index
must be rebuilt.

To detect this, we store a metadata file alongside the index
containing the embedding model name used during indexing. On load,
we compare the stored model name against the current `EMBEDDING_MODEL_NAME`.
If they differ, the index is deleted and rebuilt automatically.

In [7]:
import json

METADATA_FILE = "index_metadata.json"


def save_index_metadata(persist_dir: Path, model_name: str) -> None:
    """
    Save embedding model metadata alongside the index.

    Parameters
    ----------
    persist_dir : Path
        Directory where the index is persisted.
    model_name : str
        Name of the embedding model used for indexing.
    """
    metadata = {"embedding_model": model_name}
    metadata_path = persist_dir / METADATA_FILE
    metadata_path.write_text(json.dumps(metadata), encoding="utf-8")
    print(f"Metadata saved to {metadata_path}")


def load_and_verify_index(
    persist_dir: Path,
    collection_name: str,
    current_model_name: str,
    embed_model: HuggingFaceEmbedding,
    chunks: list[dict],
) -> VectorStoreIndex:
    """
    Load an existing index or rebuild it if the embedding model changed.

    Parameters
    ----------
    persist_dir : Path
        Directory where the index is persisted.
    collection_name : str
        Name of the ChromaDB collection.
    current_model_name : str
        Name of the current embedding model.
    embed_model : HuggingFaceEmbedding
        Initialized embedding model.
    chunks : list[dict]
        Chunks to index if rebuilding is necessary.

    Returns
    -------
    VectorStoreIndex
        Loaded or rebuilt index.
    """
    metadata_path = persist_dir / METADATA_FILE

    # Check if index exists and model is compatible
    if metadata_path.exists():
        stored = json.loads(metadata_path.read_text(encoding="utf-8"))
        stored_model = stored.get("embedding_model", "")

        if stored_model != current_model_name:
            print(f"Model mismatch: stored='{stored_model}' current='{current_model_name}'")
            print("Rebuilding index...")
            import shutil
            shutil.rmtree(persist_dir)
            persist_dir.mkdir(parents=True, exist_ok=True)
        else:
            print(f"Model match: loading existing index...")
            _, vector_store = create_vector_store(collection_name, persist_dir)
            storage_context = StorageContext.from_defaults(vector_store=vector_store)
            return VectorStoreIndex.from_vector_store(
                vector_store,
                embed_model = embed_model,
            )

    # Build new index
    _, vector_store = create_vector_store(collection_name, persist_dir)
    index = build_index(chunks, vector_store, embed_model)
    save_index_metadata(persist_dir, current_model_name)
    return index


print("Persistence functions defined ✓")

Persistence functions defined ✓


## 7. Demo — Indexing OpenStax chapter 21

This demo runs the full indexing pipeline on the carboxylic acid
derivatives chapter from OpenStax Organic Chemistry and tests
similarity search with chemistry-related queries in both English
and Spanish.

### Getting the PDF
Download the full book from
```text
https://openstax.org/details/books/organic-chemistry
```
Then extract chapter 21 (pages 741 to 792) using PyMuPDF:

```python
import fitz
doc = fitz.open("openstax_organic_chemistry.pdf")
sub = fitz.open()
sub.insert_pdf(doc, from_page=740, to_page=791)
sub.save("data/uploads/openstax_ch21_carboxylic_acid_derivatives.pdf")
```

Note: PyMuPDF uses zero-based page indexing, so page 741 corresponds to index 740.

Queries are tested in both English and Spanish to verify the multilingual
embedding model handles both languages correctly.

In [8]:
import fitz
import re

def extract_text_from_pdf(pdf_path: Path, min_block_chars: int = 20) -> str:
    """
    Extract ordered text from a PDF using block-based extraction.
    Reproduced from the ingestion pipeline notebook.
    """
    doc = fitz.open(pdf_path)
    all_text = []

    for page_num, page in enumerate(doc):
        blocks = page.get_text("blocks")
        blocks_sorted = sorted(blocks, key=lambda b: (b[1], b[0]))

        page_text = []
        for block in blocks_sorted:
            text = block[4].strip()
            if len(text) < min_block_chars:
                continue
            text = re.sub(r"\s+", " ", text)
            page_text.append(text)

        if page_text:
            all_text.append(f"--- Page {page_num + 1} ---\n" + "\n\n".join(page_text))

    doc.close()
    return "\n\n".join(all_text)


def fixed_size_chunking(
    doc: IngestedDocument,
    chunk_size: int = 512,
    chunk_overlap: int = 64,
) -> list[dict]:
    """
    Split document text into fixed-size chunks with overlap.
    Reproduced from the chunking pipeline notebook.
    """
    from llama_index.core.node_parser import SentenceSplitter
    splitter = SentenceSplitter(
        chunk_size    = chunk_size,
        chunk_overlap = chunk_overlap,
    )
    llama_doc = Document(text=doc.text)
    nodes = splitter.get_nodes_from_documents([llama_doc])

    return [
        {
            "text"     : node.text,
            "index"    : i,
            "strategy" : "fixed_size",
        }
        for i, node in enumerate(nodes)
    ]


# --- Demo ---
pdf_path = UPLOADS_DIR / "openstax_ch21_carboxylic_acid_derivatives.pdf"

doc = IngestedDocument(
    input_type  = InputType.PDF,
    text        = extract_text_from_pdf(pdf_path),
    source_path = pdf_path,
)

chunks = fixed_size_chunking(doc)
print(f"Chunks produced : {len(chunks)}")

# Build or load index
index = load_and_verify_index(
    persist_dir         = INDEX_DIR / "ch21",
    collection_name     = "openstax_ch21",
    current_model_name  = EMBEDDING_MODEL_NAME,
    embed_model         = embed_model,
    chunks              = chunks,
)

# Test queries in English and Spanish
queries = [
    "What is an amide?",
    "¿Qué es una amida?",
    "How are esters formed?",
    "¿Cómo se forman los ésteres?",
]

for query in queries:
    print(f"\nQuery: {query}")
    results = search(index, query, k=3, embed_model=embed_model)
    for i, r in enumerate(results):
        print(f"  [{i+1}] score={r['score']:.3f} | strategy={r['strategy']}")
        print(f"       {r['text'][:150]}...")

Chunks produced : 42
Model match: loading existing index...
Collection    : openstax_ch21
Persist dir   : /home/juanessao/Documents/Datos_de_Ciencia/playlists/hugging-face-hackaton/flashcard-generator-slm/data/index/ch21
Existing docs : 42

Query: What is an amide?
  [1] score=0.252 | strategy=fixed_size
       That is, an initial nucleophilic acyl substitution of an alcohol group in the enzyme on an amide linkage in the protein gives an acyl enzyme intermedi...
  [2] score=0.247 | strategy=fixed_size
       Thioesters are named like the corresponding esters. If the related ester has a common name, the prefix thio- is added to the name of the carboxylate: ...
  [3] score=0.238 | strategy=fixed_size
       Ester hydrolysis in basic solution is called saponification, after the Latin word sapo, meaning “soap.” We’ll see in Section 27.2 that soap is in fact...

Query: ¿Qué es una amida?
  [1] score=0.178 | strategy=fixed_size
       Ester hydrolysis in basic solution is called saponificati